# Lab | Chains in LangChain

## Outline

* LLMChain
* Sequential Chains
  * SimpleSequentialChain
  * SequentialChain
* Router Chain

In [55]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())


In [57]:
!pip install pandas


[notice] A new release of pip is available: 24.0 -> 25.0.1
[notice] To update, run: C:\Users\hp\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [58]:
import pandas as pd
df = pd.read_csv('Data.csv')

In [59]:
df.head()

,Product,Review
0,Queen Size Sheet Set,I ordered a king size set. My only criticism w...
1,Waterproof Phone Pouch,"I loved the waterproof sac, although the openi..."
2,Luxury Air Mattress,This mattress had a small hole in the top of i...
3,Pillows Insert,This is the best throw pillow fillers on Amazo...
4,Milk Frother Handheld\r\n,I loved this product. But they only seem to l...


## LLMChain

In [60]:
!pip install langchain_community


[notice] A new release of pip is available: 24.0 -> 25.0.1
[notice] To update, run: C:\Users\hp\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [61]:
from langchain_openai import ChatOpenAI
from langchain.prompts import ChatPromptTemplate
from langchain.chains import LLMChain

In [62]:
#Replace None by your own value and justify
llm = ChatOpenAI(temperature=0.5)


In [63]:
prompt = ChatPromptTemplate.from_template( "Write a detailed and appealing product description for the following item: {product}")

In [64]:

chain = LLMChain(llm=llm, prompt=prompt)

In [65]:
product = "Waterproof Phone Pouch"
chain.run(product)

"Introducing our innovative Waterproof Phone Pouch - the ultimate solution for keeping your phone safe and dry in any environment! Whether you're hitting the beach, going on a water adventure, or simply caught in a sudden downpour, this phone pouch has got you covered.\n\nCrafted from high-quality, durable materials, this phone pouch is designed to provide maximum protection against water, sand, dust, and debris. The secure seal ensures a watertight barrier, keeping your phone safe and dry up to 30 meters underwater. With its transparent design, you can easily access all of your phone's features and functions without having to remove it from the pouch.\n\nNot only does this phone pouch protect your device from water damage, but it also features a built-in touch-sensitive screen that allows you to use your phone while it's safely sealed inside. Whether you're taking photos, making calls, or checking messages, you can do it all without compromising the safety of your phone.\n\nThe adjust

## SimpleSequentialChain

In [66]:
from langchain.chains import SimpleSequentialChain

In [67]:
llm = ChatOpenAI(temperature=0.9)

# prompt template 1
first_prompt = ChatPromptTemplate.from_template(
    "Write a detailed and engaging product description for the following item: {product}"
)

# Chain 1
chain_one = LLMChain(llm=llm, prompt=first_prompt)

In [68]:

# prompt template 2
second_prompt = ChatPromptTemplate.from_template(
    "Based on the following product description, write a catchy marketing slogan: {input}"
)
# chain 2
chain_two = LLMChain(llm=llm, prompt=second_prompt)

In [69]:
overall_simple_chain = SimpleSequentialChain(chains=[chain_one, chain_two],
                                             verbose=True
                                            )

In [70]:
overall_simple_chain.run(product)



> Entering new SimpleSequentialChain chain...
Introducing the ultimate solution for keeping your phone safe and dry during all your outdoor adventures - the Waterproof Phone Pouch! 

Are you tired of constantly worrying about your expensive smartphone getting damaged by water, sand, or dirt while you're at the beach, pool, or hiking in the rain? Say goodbye to those concerns with this innovative and reliable waterproof pouch. 

Made from high-quality PVC material, this pouch is designed to withstand extreme weather conditions and protect your phone from water, snow, dust, and dirt. The triple zip-lock seal ensures a secure and water-tight closure, allowing you to submerge your phone up to 30 meters deep without any leaks. 

The transparent front and back panels allow you to easily use your phone's touchscreen, camera, and other functions while it remains safely sealed inside the pouch. Plus, the adjustable neck strap provides convenient hands-free carrying, so you can focus on enjoyi

'"Keep your phone high and dry with the Waterproof Phone Pouch - adventure confidently!"'

**Repeat the above twice for different products**

## SequentialChain

In [71]:
from langchain.chains import SequentialChain

In [72]:
df = pd.DataFrame({
    "Review": [
        "J’adore ce produit, la qualité est excellente !",
        "Das Produkt kam beschädigt an.",
        "Amazing product, highly recommend!",
        "لا أنصح بشراء هذا المنتج، وصل متأخرا.",
        "Très bon service client.",
        "هذا الجهاز رائع جدًا ويعمل بكفاءة عالية." 
    ]
})

In [73]:
llm = ChatOpenAI(temperature=0.9)


first_prompt = ChatPromptTemplate.from_template(
    "Translate the following product review to English:\n\n{review}"
)

chain_one = LLMChain(llm=llm, prompt=first_prompt, 
                     output_key="english_review" #Give a name to your output
                    )


In [74]:
second_prompt = ChatPromptTemplate.from_template(
    "Summarize the following product review in one sentence:\n\n{english_review}"
)

chain_two = LLMChain(llm=llm, prompt=second_prompt, 
                     output_key="summary" #give a name to this output
                    )


In [75]:
# prompt template 3: translate to english or other language
third_prompt = ChatPromptTemplate.from_template(
    "What language is this review written in?\n\n{review}"
)
# chain 3: input= Review and output= language
chain_three = LLMChain(llm=llm, prompt=third_prompt,
                       output_key="language"
                      )


In [76]:

# prompt template 4: follow up message that take as inputs the two previous prompts' variables
fourth_prompt = ChatPromptTemplate.from_template(
    "Write a follow-up message to the customer in their original language ({language}) "
    "based on the review summary: {summary}"
)
chain_four = LLMChain(llm=llm, prompt=fourth_prompt,
                      output_key="followup_message"
                     )


In [77]:
# overall_chain: input= Review 
# and output= English_Review,summary, followup_message
overall_chain = SequentialChain(
    chains=[chain_one, chain_two, chain_three, chain_four],
    input_variables=["review"],
    output_variables=["english_review", "summary", "language", "followup_message"],
    verbose=True
)

In [78]:
review = df.Review[5]
overall_chain(review)



> Entering new SequentialChain chain...

> Finished chain.


{'review': 'هذا الجهاز رائع جدًا ويعمل بكفاءة عالية.',
 'english_review': 'This device is very amazing and works with high efficiency.',
 'summary': 'The device is highly effective and impressive in its performance.',
 'language': 'Arabic',
 'followup_message': 'تمنياتنا بأن تستمر في الاستمتاع بآداء الجهاز الرائع الخاص بك! شكرا لثقتك بنا ونحن نتطلع إلى خدمتك مرة أخرى قريبا.'}

**Repeat the above twice for different products or reviews**

## Router Chain

In [79]:
physics_template = """You are a very smart physics professor. \
You are great at answering questions about physics in a concise\
and easy to understand manner. \
When you don't know the answer to a question you admit\
that you don't know.

Here is a question:
{input}"""


math_template = """You are a very good mathematician. \
You are great at answering math questions. \
You are so good because you are able to break down \
hard problems into their component parts, 
answer the component parts, and then put them together\
to answer the broader question.

Here is a question:
{input}"""

history_template = """You are a very good historian. \
You have an excellent knowledge of and understanding of people,\
events and contexts from a range of historical periods. \
You have the ability to think, reflect, debate, discuss and \
evaluate the past. You have a respect for historical evidence\
and the ability to make use of it to support your explanations \
and judgements.

Here is a question:
{input}"""


computerscience_template = """ You are a successful computer scientist.\
You have a passion for creativity, collaboration,\
forward-thinking, confidence, strong problem-solving capabilities,\
understanding of theories and algorithms, and excellent communication \
skills. You are great at answering coding questions. \
You are so good because you know how to solve a problem by \
describing the solution in imperative steps \
that a machine can easily interpret and you know how to \
choose a solution that has a good balance between \
time complexity and space complexity. 

Here is a question:
{input}"""

biology_template = """You are an excellent biologist. \
You have a deep understanding of living organisms, \
from the molecular and cellular level to entire ecosystems. \
You are skilled at observing patterns in nature, analyzing biological data, \
and explaining complex processes like evolution, genetics, physiology, and ecology. \
You can clearly communicate how life functions and adapts, \
and you make connections between different biological concepts \
to answer challenging questions.

Here is a question:
{input}"""

In [80]:
prompt_infos = [
    {
        "name": "physics", 
        "description": "Good for answering questions about physics", 
        "prompt_template": physics_template
    },
    {
        "name": "math", 
        "description": "Good for answering math questions", 
        "prompt_template": math_template
    },
    {
        "name": "History", 
        "description": "Good for answering history questions", 
        "prompt_template": history_template
    },
    {
        "name": "computer science", 
        "description": "Good for answering computer science questions", 
        "prompt_template": computerscience_template
    },
    {
        "name": "biology",
        "description": "Good for answering biology questions",
        "prompt_template": biology_template
    }
]

In [81]:
from langchain.chains.router import MultiPromptChain
from langchain.chains.router.llm_router import LLMRouterChain,RouterOutputParser
from langchain.prompts import PromptTemplate

In [82]:
llm = ChatOpenAI(temperature=0)

In [83]:
destination_chains = {}
for p_info in prompt_infos:
    name = p_info["name"]
    prompt_template = p_info["prompt_template"]
    prompt = ChatPromptTemplate.from_template(template=prompt_template)
    chain = LLMChain(llm=llm, prompt=prompt)
    destination_chains[name] = chain  
    
destinations = [f"{p['name']}: {p['description']}" for p in prompt_infos]
destinations_str = "\n".join(destinations)

In [84]:
default_prompt = ChatPromptTemplate.from_template("{input}")
default_chain = LLMChain(llm=llm, prompt=default_prompt)

In [85]:
MULTI_PROMPT_ROUTER_TEMPLATE = """Given a raw text input to a \
language model select the model prompt best suited for the input. \
You will be given the names of the available prompts and a \
description of what the prompt is best suited for. \
You may also revise the original input if you think that revising\
it will ultimately lead to a better response from the language model.

<< FORMATTING >>
Return a markdown code snippet with a JSON object formatted to look like:
```json
{{{{
    "destination": string \ name of the prompt to use or "DEFAULT"
    "next_inputs": string \ a potentially modified version of the original input
}}}}
```

REMEMBER: "destination" MUST be one of the candidate prompt \
names specified below OR it can be "DEFAULT" if the input is not\
well suited for any of the candidate prompts.
REMEMBER: "next_inputs" can just be the original input \
if you don't think any modifications are needed.

<< CANDIDATE PROMPTS >>
{destinations}

<< INPUT >>
{{input}}

<< OUTPUT (remember to include the ```json)>>"""

In [86]:
router_template = MULTI_PROMPT_ROUTER_TEMPLATE.format(
    destinations=destinations_str
)
router_prompt = PromptTemplate(
    template=router_template,
    input_variables=["input"],
    output_parser=RouterOutputParser(),
)

router_chain = LLMRouterChain.from_llm(llm, router_prompt)

In [87]:
chain = MultiPromptChain(router_chain=router_chain, 
                         destination_chains=destination_chains, 
                         default_chain=default_chain, verbose=True
                        )

In [88]:
chain.run("What is black body radiation?")



> Entering new MultiPromptChain chain...
physics: {'input': 'What is black body radiation?'}
> Finished chain.


"Black body radiation is the electromagnetic radiation emitted by a perfect absorber and emitter of radiation, known as a black body. A black body absorbs all radiation that falls on it and emits radiation across the entire electromagnetic spectrum. The spectrum of black body radiation is continuous and depends only on the temperature of the black body. This phenomenon is described by Planck's law, which states that the intensity of radiation emitted by a black body at a given wavelength is proportional to the temperature of the body and the wavelength raised to the fifth power."

In [89]:
chain.run("what is 2 + 2")



> Entering new MultiPromptChain chain...
math: {'input': 'what is 2 + 2'}
> Finished chain.


'The answer to 2 + 2 is 4.'

In [90]:
chain.run("Why does every cell in our body contain DNA?")



> Entering new MultiPromptChain chain...
biology: {'input': 'Why does every cell in our body contain DNA?'}
> Finished chain.


"Every cell in our body contains DNA because DNA is the genetic material that carries the instructions for the development, functioning, and reproduction of all living organisms. DNA contains the information needed to build and maintain an organism, including the proteins that make up our cells and tissues. \n\nHaving DNA in every cell ensures that each cell has the necessary genetic information to carry out its specific functions and to replicate itself accurately during cell division. This ensures that the genetic information is passed on to the next generation of cells, maintaining the integrity and continuity of the organism's genetic code.\n\nAdditionally, DNA is constantly being replicated and repaired in cells to ensure that genetic information is accurately transmitted and maintained. This process is essential for the proper functioning and survival of the organism."

**Repeat the above at least once for different inputs and chains executions - Be creative!**

In [102]:
saudiarabia_template = """You are an expert on Saudi Arabian history, culture, and politics. \
You have deep knowledge about the Kingdom's formation, its governance system, \
economic development, cultural heritage, and regional significance. \
You provide accurate, balanced information while respecting cultural sensitivities. \
You can discuss topics ranging from ancient history to modern Vision 2030 initiatives.

Here is a question:
{input}"""

# Add to the prompt_infos list
prompt_infos.append(
    {
        "name": "saudi arabia", 
        "description": "Good for answering questions about Saudi Arabian history, culture, and politics", 
        "prompt_template": saudiarabia_template
    }
)

# The rest of the code remains the same as before...

# Example usage:
chain.run("What were the key events in the formation of modern Saudi Arabia?")
chain.run("How has Saudi Arabia's economy evolved since the discovery of oil?")
chain.run("What are the main goals of Saudi Vision 2030?")

"The main goals of Saudi Vision 2030 are to diversify the economy away from oil dependence, create a vibrant society with a strong sense of identity and culture, and establish a thriving economy with a sustainable environment. Other goals include improving the quality of life for Saudi citizens, enhancing the country's global competitiveness, and promoting transparency and accountability in government."